In [17]:
from pathlib import Path
import json

result_path = Path("../output_gpt-5-mini")

json_files = list(result_path.glob('**/RunnerResult_DefaultRefiner.json'))
data = []
for file in json_files:
    try:
        datapoint = json.load(open(file))
        data.append(datapoint)
    except Exception as e:
        print(file)
len(data)

59

In [18]:
import pandas as pd

rows = []
for run in data:
    id = run['baseDir'].replace('/app/output/', '').replace('_', ':')
    perf = run.get("performanceTracker", {})

    for metric, events in perf.items():
        if not isinstance(events, list):
            continue
        for idx, ev in enumerate(events, start=1):
            rows.append({
                "id": id,
                "metric": metric,
                "attempt": idx,
                "start_time": ev.get("startTime"),
                "duration_ms": ev.get("duration"),
            })

perf_df = pd.DataFrame(rows)

# Optional convenience column
if not perf_df.empty:
    perf_df["duration_s"] = perf_df["duration_ms"] / 1000.0

perf_df

,id,metric,attempt,start_time,duration_ms,duration_s
0,SNYK-JS-TREEKIT-1077068,codeql.init,1,1775891338522,12401,12.401
1,SNYK-JS-TREEKIT-1077068,getExportsFromPackage,1,1775891350923,6448,6.448
2,SNYK-JS-TREEKIT-1077068,model.query,1,1775891357386,3129,3.129
3,SNYK-JS-TREEKIT-1077068,model.query,2,1775891360521,2126,2.126
4,SNYK-JS-TREEKIT-1077068,model.query,3,1775891362666,3569,3.569
...,...,...,...,...,...,...
2212,npm:moment:20170905,model.query,35,1775890104929,19636,19.636
2213,npm:moment:20170905,model.query,36,1775890124571,19476,19.476
2214,npm:moment:20170905,codeql.analyse,1,1775890144048,47703,47.703
2215,npm:moment:20170905,codeql.analyse,2,1775890191751,43202,43.202


In [19]:
# Build a runtime summary by id, then append experiment-level totals
runtime_by_id_df = (
    perf_df.groupby("id", as_index=False)["duration_ms"]
    .sum()
    .rename(columns={"duration_ms": "total_runtime_ms"})
)

total_runtime_ms = runtime_by_id_df["total_runtime_ms"].sum()
num_ids = runtime_by_id_df["id"].nunique()
avg_runtime_per_id_ms = total_runtime_ms / num_ids if num_ids else 0

summary_rows_df = pd.DataFrame([
    {"id": "__TOTAL_EXPERIMENT__", "total_runtime_ms": total_runtime_ms},
    {"id": "__AVG_PER_ID__", "total_runtime_ms": avg_runtime_per_id_ms},
])

runtime_summary_df = pd.concat([runtime_by_id_df, summary_rows_df], ignore_index=True)

# Keep seconds for numeric analysis and add a human-readable duration string
runtime_summary_df["total_runtime_s"] = runtime_summary_df["total_runtime_ms"] / 1000.0
runtime_summary_df["total_runtime_human"] = pd.to_timedelta(
    runtime_summary_df["total_runtime_ms"], unit="ms"
).astype(str)

# Drop the millisecond column from final display
runtime_summary_df = runtime_summary_df.drop(columns=["total_runtime_ms"])

runtime_summary_df

,id,total_runtime_s,total_runtime_human
0,SNYK-JS-ASSIGNDEEP-450211,545.221000,0 days 00:09:05.221000
1,SNYK-JS-CHRONONODE-1083228,2189.415000,0 days 00:36:29.415000
2,SNYK-JS-COMPONENTFLATTEN-548907,181.846000,0 days 00:03:01.846000
3,SNYK-JS-DECAL-1051028,444.048000,0 days 00:07:24.048000
4,SNYK-JS-DOTOBJECT-548905,1046.438000,0 days 00:17:26.438000
...,...,...,...
56,npm:simple-mock-server:20180226,3074.538000,0 days 00:51:14.538000
57,npm:slug:20170907,1120.753000,0 days 00:18:40.753000
58,npm:zhaolei1111:20180315,318.601000,0 days 00:05:18.601000
59,__TOTAL_EXPERIMENT__,115032.280000,1 days 07:57:12.280000
